# LoL Rank Prediction – Trénování ML modelu

Tento notebook pokrývá celý proces tvorby ML modelu pro predikci ranku hráče League of Legends.

**Postup:**
1. Načtení a prozkoumání datasetu
2. Čištění dat
3. Feature engineering – agregace na úroveň hráče
4. Explorativní analýza (EDA)
5. Škálování a příprava dat
6. Trénování modelu (LogisticRegression)
7. Evaluace – GroupKFold cross-validation
8. Analýza výsledků – confusion matrix, classification report
9. Export artefaktů

**Dataset:** ~38 000 ranked her od 2 065 hráčů (EUW), sesbíraných přes Riot Games API.
Sběr dat je popsán v samostatném notebooku `collection/lol_data_collection.ipynb`.

## 1. Instalace závislostí a importy

In [ ]:
# Instalace pro Google Colab (přeskočit pokud máte nainstalováno lokálně)
# !pip install pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
import os
import warnings

from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
)
from sklearn.base import clone

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid')

print('Importy OK')

## 2. Načtení datasetu

Dataset `lol_rank_dataset.csv` obsahuje jednotlivé ranked hry. Každý řádek = jedna hra jednoho hráče.

Data byla sesbírána z Riot Games Match V5 API – postup viz `collection/lol_data_collection.ipynb`.

In [ ]:
# Cesta k datasetu – upravte podle umístění
DATASET_PATH = 'datasets/lol_rank_dataset.csv'

# Pro Google Colab:
# from google.colab import files
# uploaded = files.upload()  # nahrajte lol_rank_dataset.csv
# DATASET_PATH = 'lol_rank_dataset.csv'

raw_df = pd.read_csv(DATASET_PATH, low_memory=False)
print(f'Načteno: {len(raw_df)} řádků, {len(raw_df.columns)} sloupců')
print(f'Sloupce: {list(raw_df.columns)}')
raw_df.head()

In [ ]:
# Základní info o datasetu
print(f'Unikátních hráčů: {raw_df["puuid"].nunique()}')
print(f'Unikátních zápasů: {raw_df["matchId"].nunique()}')
print(f'Unikátních championů: {raw_df["championName"].nunique()}')
print(f'\nRozložení tierů (počet her):')
print(raw_df['tier'].value_counts().sort_index())
print(f'\nChybějící hodnoty:')
print(raw_df.isnull().sum()[raw_df.isnull().sum() > 0])

## 3. Čištění dat

Kroky čištění:
- Odstranění řádků bez `puuid` nebo `tier`
- Filtrování pouze platných tierů (Iron–Diamond)
- Konverze numerických sloupců (řeší mixed types z CSV)
- Odstranění řádků s chybějícími klíčovými hodnotami
- Odstranění her kratších než 5 minut (remaky)

In [ ]:
VALID_TIERS = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND']

def load_and_clean(df):
    """Vyčistí surový dataset."""
    df = df.copy()
    
    # Odstranění řádků bez puuid nebo tier
    before = len(df)
    df = df.dropna(subset=['puuid', 'tier'])
    print(f'Odstraněno {before - len(df)} řádků bez puuid/tier')
    
    # Jen platné tiery
    before = len(df)
    df = df[df['tier'].isin(VALID_TIERS)]
    print(f'Odstraněno {before - len(df)} řádků s neplatným tierem')
    
    # Konverze numerických sloupců
    numeric_cols = [
        'kills', 'deaths', 'assists', 'totalMinionsKilled', 'neutralMinionsKilled',
        'visionScore', 'totalDamageDealtToChampions', 'totalDamageTaken',
        'goldEarned', 'timePlayed', 'kda', 'cs_per_min', 'damage_per_min',
        'gold_per_min', 'deaths_per_min', 'gameDuration',
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Odstranění řádků s chybějícími klíčovými hodnotami
    before = len(df)
    df = df.dropna(subset=['kills', 'deaths', 'assists', 'timePlayed'])
    print(f'Odstraněno {before - len(df)} řádků s chybějícími KDA/timePlayed')
    
    # Hry kratší než 5 minut = remaky
    before = len(df)
    df = df[df['timePlayed'] > 300]
    print(f'Odstraněno {before - len(df)} her kratších než 5 minut (remaky)')
    
    return df

df = load_and_clean(raw_df)
print(f'\nPo čištění: {len(df)} řádků')
print(f'Unikátních hráčů: {df["puuid"].nunique()}')

## 4. Feature Engineering – agregace na úroveň hráče

Model nepredikuje z jedné hry, ale z **profilu hráče**. Z N her se spočítají:
- **Průměry** (mean) – kills, deaths, assists, KDA, CS/min, damage/min, gold/min, deaths/min, vision/min, dmg taken/min
- **Směrodatné odchylky** (std) – konzistence hráče
- **Mediány** – robustní střed pro KDA, CS, damage, gold, deaths, vision
- **Meta features** – winrate, počet unikátních rolí, počet unikátních championů, průměrná délka hry

Celkem **30 features** na hráče.

In [ ]:
# Definice feature skupin
MEAN_FEATURES = [
    'kills', 'deaths', 'assists', 'kda',
    'cs_per_min', 'damage_per_min', 'gold_per_min', 'deaths_per_min',
    'vision_per_min', 'damage_taken_per_min',
]

STD_FEATURES = [
    'kda', 'cs_per_min', 'damage_per_min', 'gold_per_min',
    'deaths_per_min', 'vision_per_min', 'damage_taken_per_min',
    'kills', 'deaths', 'assists',
]

MEDIAN_FEATURES = [
    'kda', 'cs_per_min', 'damage_per_min', 'gold_per_min',
    'deaths_per_min', 'vision_per_min',
]

TIER_ORDER = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND']
TIER_TO_INT = {t: i for i, t in enumerate(TIER_ORDER)}


def compute_derived_features(df):
    """Dopočítá odvozené per-minutové metriky."""
    df = df.copy()
    game_minutes = df['timePlayed'] / 60.0
    game_minutes = game_minutes.replace(0, np.nan)
    df['vision_per_min'] = df['visionScore'] / game_minutes
    df['damage_taken_per_min'] = df['totalDamageTaken'] / game_minutes
    return df


def aggregate_player_games(games_df):
    """Agreguje hry jednoho hráče do jednoho feature vektoru."""
    result = {}
    for f in MEAN_FEATURES:
        result[f'{f}_mean'] = games_df[f].mean() if f in games_df.columns else 0
    for f in STD_FEATURES:
        if f in games_df.columns:
            val = games_df[f].std()
            result[f'{f}_std'] = val if pd.notna(val) else 0
        else:
            result[f'{f}_std'] = 0
    for f in MEDIAN_FEATURES:
        result[f'{f}_median'] = games_df[f].median() if f in games_df.columns else 0
    result['winrate'] = games_df['win'].mean() if 'win' in games_df.columns else 0
    result['unique_roles'] = games_df['role'].nunique() if 'role' in games_df.columns else 1
    result['unique_champions'] = games_df['championName'].nunique() if 'championName' in games_df.columns else 1
    if 'gameDuration' in games_df.columns:
        result['avg_game_duration'] = games_df['gameDuration'].mean()
    elif 'timePlayed' in games_df.columns:
        result['avg_game_duration'] = games_df['timePlayed'].mean()
    else:
        result['avg_game_duration'] = 0
    return result


def build_player_dataset(df):
    """Sestaví dataset na úrovni hráčů z per-game dat."""
    df = compute_derived_features(df)
    players = []
    for puuid, group in df.groupby('puuid'):
        agg = aggregate_player_games(group)
        agg['puuid'] = puuid
        agg['tier'] = group['tier'].iloc[0]
        agg['tier_encoded'] = TIER_TO_INT.get(group['tier'].iloc[0], -1)
        players.append(agg)
    result = pd.DataFrame(players)
    result = result.fillna(0)
    return result


# Seznam všech feature jmen
FEATURE_NAMES = (
    [f'{f}_mean' for f in MEAN_FEATURES] +
    [f'{f}_std' for f in STD_FEATURES] +
    [f'{f}_median' for f in MEDIAN_FEATURES] +
    ['winrate', 'unique_roles', 'unique_champions', 'avg_game_duration']
)

print(f'Feature names ({len(FEATURE_NAMES)}): {FEATURE_NAMES}')

In [ ]:
# Sestavení hráčského datasetu
player_df = build_player_dataset(df)

print(f'Počet hráčů: {len(player_df)}')
print(f'\nRozložení tierů:')
print(player_df['tier'].value_counts().reindex(TIER_ORDER))
print(f'\nPrvní 3 řádky:')
player_df.head(3)

## 5. Explorativní analýza (EDA)

Vizualizace rozložení dat a korelací mezi features a tierem.

In [ ]:
# Rozložení hráčů podle tieru
fig, ax = plt.subplots(figsize=(10, 4))
tier_colors = ['#6B5B50', '#CD7F32', '#A8B4C0', '#FFD700', '#00CED1', '#50C878', '#B9F2FF']
counts = player_df['tier'].value_counts().reindex(TIER_ORDER)
ax.bar(counts.index, counts.values, color=tier_colors, edgecolor='black', linewidth=0.5)
ax.set_title('Počet hráčů podle tieru', fontsize=14, fontweight='bold')
ax.set_ylabel('Počet hráčů')
for i, v in enumerate(counts.values):
    ax.text(i, v + 3, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Jak se klíčové metriky liší mezi tiery
key_metrics = ['kda_mean', 'cs_per_min_mean', 'damage_per_min_mean', 
               'gold_per_min_mean', 'vision_per_min_mean', 'winrate']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, metric in zip(axes.flatten(), key_metrics):
    data_by_tier = [player_df[player_df['tier'] == t][metric].values for t in TIER_ORDER]
    bp = ax.boxplot(data_by_tier, labels=TIER_ORDER, patch_artist=True)
    for patch, color in zip(bp['boxes'], tier_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Rozložení klíčových metrik podle tieru', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Korelační matice features vs. tier
feature_cols = [f for f in FEATURE_NAMES if f in player_df.columns]
corr_with_tier = player_df[feature_cols + ['tier_encoded']].corr()['tier_encoded'].drop('tier_encoded')
corr_sorted = corr_with_tier.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#22c55e' if v > 0 else '#ef4444' for v in corr_sorted.values]
ax.barh(corr_sorted.index, corr_sorted.values, color=colors, edgecolor='black', linewidth=0.3)
ax.set_xlabel('Korelace s tierem (vyšší = lepší rank)')
ax.set_title('Korelace features s tierem', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Top 5 pozitivně korelované:')
print(corr_sorted.head(5))
print('\nTop 5 negativně korelované:')
print(corr_sorted.tail(5))

## 6. Příprava dat pro trénování

- **StandardScaler** – škálování features na nulový průměr a jednotkový rozptyl
- **GroupKFold** – zajistí, že hry stejného hráče jsou vždy ve stejném foldu (žádný data leakage)

In [ ]:
# Příprava X, y, groups
feature_names = [f for f in FEATURE_NAMES if f in player_df.columns]
X = player_df[feature_names].values
y = player_df['tier_encoded'].values
groups = player_df['puuid'].values

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Počet tříd: {len(np.unique(y))}')
print(f'Rozložení tříd: {dict(zip(TIER_ORDER, np.bincount(y)))}')

In [ ]:
# Škálování
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Škálování hotovo')
print(f'Průměr po škálování (prvních 5 features): {X_scaled.mean(axis=0)[:5].round(6)}')
print(f'Std po škálování (prvních 5 features): {X_scaled.std(axis=0)[:5].round(4)}')

## 7. Trénování modelu – LogisticRegression

Logistická regrese je zvolena jako baseline model:
- Interpretovatelná (koeficienty = feature importance)
- Rychlý trénink
- Dobře funguje na tabulkových datech se StandardScalerem

Validace přes **GroupKFold (5 foldů)** – hráči nikdy nejsou rozřezáni mezi train a test.

In [ ]:
# Definice modelu
model = LogisticRegression(max_iter=2000, C=1.0)

# Cross-validation se GroupKFold
gkf = GroupKFold(n_splits=5)
scores = cross_val_score(model, X_scaled, y, cv=gkf, groups=groups, scoring='accuracy')

print('=== Cross-Validation Výsledky ===')
for i, s in enumerate(scores):
    print(f'  Fold {i+1}: {s:.4f}')
print(f'\nPrůměrná CV accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})')
print(f'Random baseline (7 tříd): {1/7:.4f} = {1/7*100:.1f}%')
print(f'Model je {scores.mean() / (1/7):.1f}× lepší než náhoda')

In [ ]:
# Trénování finálního modelu na všech datech
model.fit(X_scaled, y)
print('Finální model natrénován na všech datech')
print(f'Koeficienty shape: {model.coef_.shape}')  # (7 tříd × 30 features)

## 8. Evaluace modelu

### 8.1 Classification Report (cross-validated)

In [ ]:
# Predikce přes GroupKFold (out-of-fold predikce pro každého hráče)
all_preds = np.zeros_like(y)
fold_accuracies = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_scaled, y, groups)):
    model_clone = clone(model)
    model_clone.fit(X_scaled[train_idx], y[train_idx])
    all_preds[test_idx] = model_clone.predict(X_scaled[test_idx])
    fold_acc = accuracy_score(y[test_idx], all_preds[test_idx])
    fold_accuracies.append(fold_acc)

print(f'Overall CV Accuracy: {accuracy_score(y, all_preds):.4f}')
print(f'\nClassification Report:')
print(classification_report(y, all_preds, target_names=TIER_ORDER))

### 8.2 Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y, all_preds)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Absolutní hodnoty
disp1 = ConfusionMatrixDisplay(cm, display_labels=TIER_ORDER)
disp1.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (absolutní)', fontsize=13, fontweight='bold')

# Normalizovaná (procenta)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp2 = ConfusionMatrixDisplay(cm_norm, display_labels=TIER_ORDER)
disp2.plot(ax=axes[1], cmap='Oranges', values_format='.2f')
axes[1].set_title('Confusion Matrix (normalizovaná)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Analýza diagonály vs. sousedních tierů
exact = np.trace(cm)
within_1 = sum(cm[i][j] for i in range(len(TIER_ORDER)) for j in range(len(TIER_ORDER)) if abs(i - j) <= 1)
total = cm.sum()

print(f'Přesná shoda: {exact}/{total} = {exact/total*100:.1f}%')
print(f'Shoda ± 1 tier: {within_1}/{total} = {within_1/total*100:.1f}%')

### 8.3 Feature Importance

In [ ]:
# Feature importance (průměr absolutních koeficientů přes všechny třídy)
avg_coef = np.mean(np.abs(model.coef_), axis=0)
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': avg_coef
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(importance_df['feature'], importance_df['importance'], color='#3273fa', edgecolor='black', linewidth=0.3)
ax.set_xlabel('Průměrný absolutní koeficient')
ax.set_title('Feature Importance (LogisticRegression)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Top 10 nejdůležitějších features:')
print(importance_df.tail(10).to_string(index=False))

### 8.4 Per-tier analýza – průměrné profily

In [ ]:
# Průměrné hodnoty klíčových metrik per tier
tier_means = player_df.groupby('tier')[key_metrics].mean().reindex(TIER_ORDER)

fig, ax = plt.subplots(figsize=(12, 6))
tier_means.plot(kind='bar', ax=ax, width=0.8, edgecolor='black', linewidth=0.3)
ax.set_title('Průměrné hodnoty metrik podle tieru', fontsize=14, fontweight='bold')
ax.set_xlabel('Tier')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print('Průměrné hodnoty per tier:')
print(tier_means.round(4).to_string())

## 9. Výpočet statistik ranků (pro analytiku v aplikaci)

In [ ]:
def compute_rank_stats(player_df, feature_names):
    """Spočítá průměr, std a medián features pro každý tier."""
    stats = {}
    for tier in TIER_ORDER:
        tier_data = player_df[player_df['tier'] == tier][feature_names]
        stats[tier] = {
            'mean': tier_data.mean().to_dict(),
            'std': tier_data.std().to_dict(),
            'median': tier_data.median().to_dict(),
            'count': len(tier_data),
        }
    return stats

rank_stats = compute_rank_stats(player_df, feature_names)
print('Rank stats spočítány')
for tier in TIER_ORDER:
    print(f'  {tier}: {rank_stats[tier]["count"]} hráčů')

## 10. Export artefaktů modelu

Uložení natrénovaného modelu, scaleru, rank statistik a metadat pro použití v aplikaci.

In [ ]:
MODELS_DIR = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Model
with open(os.path.join(MODELS_DIR, 'model.pkl'), 'wb') as f:
    pickle.dump(model, f)
print(f'Model uložen: {os.path.join(MODELS_DIR, "model.pkl")}')

# Scaler
with open(os.path.join(MODELS_DIR, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print(f'Scaler uložen: {os.path.join(MODELS_DIR, "scaler.pkl")}')

# Rank statistiky
with open(os.path.join(MODELS_DIR, 'rank_stats.pkl'), 'wb') as f:
    pickle.dump(rank_stats, f)
print(f'Rank stats uloženy: {os.path.join(MODELS_DIR, "rank_stats.pkl")}')

# Metadata
meta = {
    'model_name': 'LogisticRegression',
    'feature_names': feature_names,
    'tier_order': TIER_ORDER,
    'cv_accuracy': float(scores.mean()),
    'num_players': len(player_df),
}
with open(os.path.join(MODELS_DIR, 'model_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Metadata uložena: {os.path.join(MODELS_DIR, "model_meta.json")}')

print(f'\nVšechny artefakty uloženy do složky: {MODELS_DIR}/')

## 11. Shrnutí

| Parametr | Hodnota |
|---|---|
| Model | LogisticRegression (C=1.0, max_iter=2000) |
| Preprocessing | StandardScaler |
| Validace | GroupKFold (5 foldů, grupováno podle hráče) |
| Features | 30 (mean, std, median agregace + meta) |
| Třídy | 7 (Iron – Diamond) |
| Dataset | ~2065 hráčů, ~38 000 her |
| Zdroj dat | Riot Games Match V5 API (EUW server) |

Model dosahuje ~40% přesnosti na 7-třídním problému (vs. 14.3% random baseline),
a přesnosti **± 1 tier** kolem 70%+, což odpovídá plynulé hranici mezi sousedními ranky.